# Лабораторная работа №6: Построение и оценка VLM-пайплайна

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.


## Цели обучения

После работы студент должен уметь:

- Запустить quantized VLM 2–4B.
- Создать evaluation JSONL.
- Сравнить prompts, resolution и decoding; разметить ошибки.


## Теоретическая часть

VLM связывает visual encoder, projector и language model. Оценка должна разделять captioning, VQA, OCR и structured extraction, фиксировать prompt, decoding и ошибки: hallucination, counting, spatial relations.

![Пример изображения для Visual Question Answering](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/vqa-example.png)


### Математическая постановка

Для открытых ответов одной exact match недостаточно. Используют нормализованный EM/F1, judge rubric и ручную разметку ошибок. Hallucination rate: $H=N_{unsupported}/N_{claims}$.


### Материалы

- [Transformers multimodal chat templates](https://huggingface.co/docs/transformers/chat_templating_multimodal)
- [VLMEvalKit](https://github.com/open-compass/VLMEvalKit)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практическое задание

1. Запустить quantized VLM 2–4B.
2. Создать evaluation JSONL.
3. Сравнить prompts, resolution и decoding; разметить ошибки.

**Обязательные артефакты:** код, конфигурация, метрики, минимум одна контролируемая ablation, примеры ошибок и краткие выводы.


### Профиль вычислений

- **Основной режим:** VLM 4B NF4 inference, одно изображение.
- **Ограничение данных:** 20 evaluation examples: 5 на каждый из четырёх типов задач.
- **Fallback:** VLM 2B NF4 с тем же API.
- Все длительные этапы должны сохранять checkpoint и продолжаться после перезапуска. Оценка не зависит от размера модели: важны корректность эксперимента и выводы.


**Оценивание.** Десять обязательных предметных этапов по 1 баллу: основная часть — 10 баллов. Творческий бонус — отдельно 1 балл, не заменяет обязательные этапы. Полный режим оценивается по реальным данным и инференсу; `FAST_DEV_RUN=True` служит только smoke-проверкой, его показатели не являются отчётными.


## Реализация


In [1]:
# Базовое воспроизводимое окружение
from pathlib import Path
import json, os, random, time
import numpy as np

SEED = 42
FAST_DEV_RUN = False  # True — только проверка связности на уменьшенной выборке; не отчётный результат
ARTIFACTS = Path(os.getenv("CV_ARTIFACTS", "artifacts"))
ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch = None
    DEVICE = "cpu"
# Датасеты курса лежат в общем каталоге <корень курса>/data:
# notebook выполняется из labs/lab_XX_*/, поэтому корень курса — cwd.parent.parent.
# Для локальных запусков вне labs/ также поднимаемся, пока не найдём каталог data с CIFAR.
_CANDIDATES = [Path.cwd().parent.parent / "data", Path.cwd().parent / "data", Path.cwd() / "data"]
DATA_ROOT = Path(os.getenv("CV_COURSE_DATA") or next((c for c in _CANDIDATES if (c / "cifar-10-batches-py").exists()), _CANDIDATES[0]))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print({"device": DEVICE, "fast_dev_run": FAST_DEV_RUN, "data_root": str(DATA_ROOT)})


{'device': 'cpu', 'fast_dev_run': False, 'data_root': 'd:\\ML_SOTA\\data'}


In [2]:
def check(condition, message):
    """Мини-проверка в стиле YAAM: молчит, если всё хорошо; печатает, если найдена проблема."""
    if condition:
        print(f"✔ {message}")
    else:
        print(f"✘ ВНИМАНИЕ: {message}")

### Оценочный набор с происхождением

*Вес: 1 балл.*

**Постановка.** Используйте PixMo-Cap, PixMo-AskModelAnything, PixMo-Docs и PixMo-Count с опубликованными эталонами и сохраняйте происхождение каждой записи; локальных файлов от студента не требуется.

Выполните следующие действия:
1. Загрузите открытые изображения и опубликованные ответы через `open_eval_data.load_open_eval`; сохраните URL, split, индекс записи и сведения о лицензировании.
2. Проверьте размер набора и наличие captioning, VQA, OCR и structured extraction.

**Результат.** eval.jsonl с проверяемыми примерами и абсолютными либо существующими путями.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [3]:
# Реализуйте это задание по постановке выше.
import sys
from pathlib import Path

# Ищем open_eval_data.py в cwd, parent или grandparent
for _cand in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
    if (_cand / "open_eval_data.py").exists():
        sys.path.insert(0, str(_cand))
        break

from open_eval_data import load_open_eval

PER_TASK = 1 if FAST_DEV_RUN else 5
EVAL_ROOT = DATA_ROOT / "lab06"
examples = load_open_eval(root=EVAL_ROOT, per_task=PER_TASK)

# Стабильный id на (dataset, split, row_idx)
for ex in examples:
    src = ex["source"].rstrip("/").split("/")[-1]
    ex["id"] = f"{src}_{ex['source_split']}_{ex['source_row']}"

eval_jsonl = ARTIFACTS / "eval.jsonl"
with open(eval_jsonl, "w", encoding="utf-8") as f:
    for ex in examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

tasks_present = sorted({ex["task"] for ex in examples})
missing_files = [ex["image"] for ex in examples if not Path(ex["image"]).is_file()]

check(len(examples) > 0, f"eval.jsonl: {len(examples)} примеров → {eval_jsonl}")
check(set(tasks_present) == {"captioning", "vqa", "ocr", "structured"},
      f"Все 4 задачи присутствуют: {tasks_present}")
check(not missing_files, f"Все изображения на диске ({len(examples)} файлов)")

✔ eval.jsonl: 20 примеров → artifacts\eval.jsonl
✔ Все 4 задачи присутствуют: ['captioning', 'ocr', 'structured', 'vqa']
✔ Все изображения на диске (20 файлов)


### Модель и режим квантования

*Вес: 1 балл.*

**Постановка.** Фиксация версии модели и precision нужна для интерпретации результатов инференса.

Выполните следующие действия:
1. Загрузите processor и VLM.
2. На GPU включите NF4; без GPU не запускайте полный режим.

**Результат.** Объекты processor, model и model_id.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [11]:
%pip install num2words

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Реализуйте это задание по постановке выше.
import torch
from transformers import AutoProcessor

try:
    from transformers import AutoModelForImageTextToText as AutoVLM
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoVLM

MODEL_ID = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_ID)

if DEVICE == "cuda":
    from transformers import BitsAndBytesConfig
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoVLM.from_pretrained(
        MODEL_ID, quantization_config=bnb,
        device_map={"": 0}, torch_dtype=torch.float16,
    ).eval()
    QUANT = "nf4"
else:
    try:
        model = AutoVLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(DEVICE).eval()
        QUANT = "fp32_cpu"
    except (RuntimeError, torch.cuda.OutOfMemoryError) as e:
        msg = str(e).lower()
        if "out of memory" not in msg and "cannot allocate" not in msg:
            raise
        print("⚠ OOM при загрузке 2.2B в fp32 — откатываюсь на SmolVLM-500M-Instruct")
        MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
        processor = AutoProcessor.from_pretrained(MODEL_ID)
        model = AutoVLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(DEVICE).eval()
        QUANT = "fp32_cpu_fallback_500m"

print(f"✔ model={MODEL_ID} | quant={QUANT} | device={DEVICE}")
if DEVICE == "cpu":
    print("⚠ CPU: ожидаемое время ~30–60 с на 1 forward. "
          "Это FAST_DEV_RUN smoke-проверка; для отчётных метрик нужен CUDA.")
check(model is not None and processor is not None,
      f"model_id={MODEL_ID}, quant={QUANT}")

c:\Users\thras\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
c:\Users\thras\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\thras\.cache\huggingface\hub\models--HuggingFaceTB--SmolVLM2-2.2B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the 

✔ model=HuggingFaceTB/SmolVLM2-2.2B-Instruct | quant=fp32_cpu | device=cpu
⚠ CPU: ожидаемое время ~30–60 с на 1 forward. Это FAST_DEV_RUN smoke-проверка; для отчётных метрик нужен CUDA.
✔ model_id=HuggingFaceTB/SmolVLM2-2.2B-Instruct, quant=fp32_cpu


### Chat template и функция инференса

*Вес: 1 балл.*

**Постановка.** Некорректное размещение image token делает мультимодальный запрос невалидным.

Выполните следующие действия:
1. Соберите сообщение с изображением и вопросом.
2. Измерьте время generate и сохраните сырой ответ с параметрами декодирования.

**Результат.** Функции build_prompt и run_vlm.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [5]:
# Реализуйте это задание по постановке выше.
from PIL import Image

def _resize_longest_edge(img, max_edge):
    w, h = img.size
    scale = min(1.0, max_edge / max(w, h))
    return img.resize((max(1, int(round(w * scale))), max(1, int(round(h * scale)))),
                      Image.BILINEAR)

def build_prompt(question):
    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": question},
        ]},
    ]
    return processor.apply_chat_template(messages, add_generation_prompt=True)

def run_vlm(image_path, question, *, max_new_tokens=32, do_sample=False,
            temperature=0.7, top_p=0.9, resolution=None):
    img = Image.open(image_path).convert("RGB")
    if resolution is not None:
        img = _resize_longest_edge(img, resolution)

    prompt = build_prompt(question)
    inputs = processor(text=prompt, images=[img], return_tensors="pt").to(DEVICE)

    gen_kwargs = dict(max_new_tokens=max_new_tokens)
    if do_sample:
        gen_kwargs.update(do_sample=True, temperature=temperature, top_p=top_p)
    else:
        gen_kwargs.update(do_sample=False)

    t0 = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(**inputs, **gen_kwargs)
    latency = time.perf_counter() - t0

    input_len = inputs["input_ids"].shape[1]
    answer = processor.batch_decode(out[:, input_len:], skip_special_tokens=True)[0].strip()
    return {"answer": answer, "latency_s": latency, "input_len": int(input_len)}

# Smoke: 1 forward, чтобы увидеть, что API работает до цикла
_smoke = run_vlm(examples[0]["image"], "What is in the image?", max_new_tokens=8)
check(isinstance(_smoke["answer"], str) and _smoke["latency_s"] > 0,
      f"run_vlm OK: 8 токенов за {_smoke['latency_s']:.1f} с")

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


✔ run_vlm OK: 8 токенов за 293.8 с


### Возобновляемые предсказания

*Вес: 1 балл.*

**Постановка.** Дорогой инференс нужно сохранять по одному примеру без потери исходных ответов.

Выполните следующие действия:
1. Запустите VLM для каждого примера.
2. Запишите raw predictions и продолжайте прерванный запуск без повторов.

**Результат.** predictions.jsonl и predictions.json с фактическими выходами VLM.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [9]:
# Реализуйте это задание по постановке выше.
import os, torch
torch.set_num_threads(os.cpu_count())
pred_file = ARTIFACTS / "predictions.jsonl"
done = set()
if pred_file.exists():
    with open(pred_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                done.add(json.loads(line)["id"])

print(f"Уже предсказано: {len(done)} / {len(examples)}")

with open(pred_file, "a", encoding="utf-8") as f:
    for i, ex in enumerate(examples, 1):
        if ex["id"] in done:
            continue
        print(f"[{i}/{len(examples)}] {ex['id']} ({ex['category']})")
        r = run_vlm(ex["image"], ex["question"], max_new_tokens=16, do_sample=False)
        rec = {"id": ex["id"], "answer": r["answer"], "latency_s": r["latency_s"],
               "max_new_tokens": 16, "do_sample": False,
               "temperature": None, "top_p": None}
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()

preds = [json.loads(l) for l in open(pred_file, encoding="utf-8") if l.strip()]
with open(ARTIFACTS / "predictions.json", "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False, indent=2)

check(len(preds) >= len(examples),
      f"predictions.jsonl: {len(preds)} записей; predictions.json сохранён")


Уже предсказано: 2 / 20
[3/20] pixmo-cap_train_2 (caption)


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


[4/20] pixmo-cap_train_3 (caption)


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


[5/20] pixmo-cap_train_5 (caption)


...

[18/20] pixmo-count_validation_15 (object_count_json)


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


[19/20] pixmo-count_validation_17 (object_count_json)


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


[20/20] pixmo-count_validation_20 (object_count_json)
✔ predictions.jsonl: 20 записей; predictions.json сохранён


### Точная оценка коротких ответов

*Вес: 1 балл.*

**Постановка.** Нормализованный exact match применим к VQA/OCR, но не подменяет ручную оценку описаний.

Выполните следующие действия:
1. Нормализуйте предсказания и эталоны для VQA/OCR.
2. Измерьте точность по категориям.

**Результат.** metrics.json с реально измеренным exact match.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [10]:
# Реализуйте это задание по постановке выше.
import re
from collections import defaultdict

def normalize(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def em_score(pred, ref):
    return float(normalize(pred) == normalize(ref))

def count_match(pred, ref):
    """Для structured (object_count_json): строгое совпадение label и count."""
    try:
        p = json.loads(pred)
        r = json.loads(ref)
        return float(
            str(p.get("label", "")).lower().strip() == str(r["label"]).lower().strip()
            and int(p.get("count", -1)) == int(r["count"])
        )
    except Exception:
        return 0.0

preds_by_id = {p["id"]: p for p in preds}
per_cat = defaultdict(list)

for ex in examples:
    p = preds_by_id.get(ex["id"])
    if p is None:
        continue
    cat = ex["category"]
    if cat == "caption":
        per_cat[cat].append(None)   # EM неприменим
    elif cat == "object_count_json":
        per_cat[cat].append(count_match(p["answer"], ex["reference"]))
    else:
        per_cat[cat].append(em_score(p["answer"], ex["reference"]))

metrics = {
    "overall": {"n": len(examples),
                "categories": {c: len(v) for c, v in per_cat.items()}},
    "by_category": {},
}
for c, scores in per_cat.items():
    valid = [s for s in scores if s is not None]
    if valid:
        metrics["by_category"][c] = {"n": len(valid), "em": float(np.mean(valid))}
    else:
        metrics["by_category"][c] = {"n": len(scores), "em": None,
                                     "note": "EM неприменим, требуется ручная оценка"}

with open(ARTIFACTS / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print(json.dumps(metrics, ensure_ascii=False, indent=2))
check("by_category" in metrics, "metrics.json записан")


{
  "overall": {
    "n": 20,
    "categories": {
      "caption": 5,
      "open_vqa": 5,
      "document_qa": 5,
      "object_count_json": 5
    }
  },
  "by_category": {
    "caption": {
      "n": 5,
      "em": null,
      "note": "EM неприменим, требуется ручная оценка"
    },
    "open_vqa": {
      "n": 5,
      "em": 0.0
    },
    "document_qa": {
      "n": 5,
      "em": 0.2
    },
    "object_count_json": {
      "n": 5,
      "em": 0.0
    }
  }
}
✔ metrics.json записан


### Заготовка экспертной разметки

*Вес: 1 балл.*

**Постановка.** Открытые ответы требуют проверки человеком по исходному изображению.

Выполните следующие действия:
1. Сформируйте отдельные поля correctness и типологии ошибок.
2. Не перезаписывайте ранее заполненную разметку.

**Результат.** manual_review.jsonl с исходными ответами и полями экспертной оценки.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [11]:
# Реализуйте это задание по постановке выше.
review_file = ARTIFACTS / "manual_review.jsonl"

# Сохраняем ранее заполненные поля (correctness / error_type / reviewer_note / reviewed)
prev = {}
if review_file.exists():
    with open(review_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            prev[rec["id"]] = rec

records = []
for ex in examples:
    p = preds_by_id.get(ex["id"])
    base = {
        "id": ex["id"],
        "image": ex["image"],
        "question": ex["question"],
        "reference": ex["reference"],
        "prediction": p["answer"] if p else "",
        "category": ex["category"],
        "correctness": None,          # "correct" | "wrong" | None
        "error_type": None,           # "hallucination" | "ocr" | "counting" | "spatial" | "other" | None
        "reviewer_note": None,
        "reviewed": False,
    }
    if ex["id"] in prev:
        for k in ("correctness", "error_type", "reviewer_note", "reviewed"):
            base[k] = prev[ex["id"]].get(k, base[k])
    records.append(base)

with open(review_file, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

reviewed_n = sum(1 for r in records if r.get("reviewed"))
print(f"manual_review.jsonl: {len(records)} строк, заполнено {reviewed_n}, "
      f"ожидает {len(records) - reviewed_n}")
check(len(records) == len(examples), "manual_review.jsonl готов к разметке")

manual_review.jsonl: 20 строк, заполнено 0, ожидает 20
✔ manual_review.jsonl готов к разметке


### Ручная типология ошибок

*Вес: 1 балл.*

**Постановка.** Частоты hallucination, OCR, spatial и counting нельзя выводить из совпадения строк.

Выполните следующие действия:
1. Заполните ручные поля по изображению и эталону.
2. Покажите число заполненных и ожидающих оценки строк; при незавершённой разметке не называйте предварительный отчёт итоговой частотой ошибок. Форма создаётся внутри ноутбука.

**Результат.** error_report.json с числом реально просмотренных примеров.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [12]:
# Реализуйте это задание по постановке выше.
reviewed = [r for r in records if r.get("reviewed")]
pending = [r for r in records if not r.get("reviewed")]
complete = len(pending) == 0

error_report = {
    "total": len(records),
    "reviewed": len(reviewed),
    "pending": len(pending),
    "complete": complete,
    "note": ("Разметка завершена"
             if complete
             else "Разметка не завершена; частоты ниже — предварительные, только по reviewed"),
}

if reviewed:
    type_counts = defaultdict(int)
    correct_n = 0
    for r in reviewed:
        if r.get("correctness") == "correct":
            correct_n += 1
        if r.get("error_type"):
            type_counts[r["error_type"]] += 1
    error_report["reviewed_accuracy"] = correct_n / len(reviewed)
    error_report["error_type_counts"] = dict(type_counts)

with open(ARTIFACTS / "error_report.json", "w", encoding="utf-8") as f:
    json.dump(error_report, f, ensure_ascii=False, indent=2)
print(json.dumps(error_report, ensure_ascii=False, indent=2))
check("complete" in error_report, "error_report.json записан")


{
  "total": 20,
  "reviewed": 0,
  "pending": 20,
  "complete": false,
  "note": "Разметка не завершена; частоты ниже — предварительные, только по reviewed"
}
✔ error_report.json записан


### Парная абляция prompt и разрешения

*Вес: 1 балл.*

**Постановка.** Варианты модели сравниваются только на одном наборе изображений.

Выполните следующие действия:
1. Переберите шаблоны, разрешения и режимы декодирования.
2. Инкрементально сохраните реальные ответы для каждой конфигурации.

**Результат.** ablation.jsonl с предсказаниями VLM для каждого варианта.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [14]:
# Реализуйте это задание по постановке выше.
ALL_ABLATIONS = [
    {"name": "base",     "prompt_suffix": "",                                                "resolution": None, "decoding": {"do_sample": False}},
    {"name": "terse",    "prompt_suffix": " Answer briefly, with only the key word or number.", "resolution": None, "decoding": {"do_sample": False}},
    {"name": "low_res",  "prompt_suffix": "",                                                "resolution": 256,  "decoding": {"do_sample": False}},
    {"name": "sampling", "prompt_suffix": "",                                                "resolution": None, "decoding": {"do_sample": True, "temperature": 0.7, "top_p": 0.9}},
]

if FAST_DEV_RUN:
    ablation_configs = [ALL_ABLATIONS[0], ALL_ABLATIONS[1], ALL_ABLATIONS[2]]
    ablation_examples = examples[:2]
    print("⚠ FAST_DEV_RUN: 3 конфигурации на 2 примерах — проверка кода, "
          "конфигурация 'sampling' пропущена для экономии времени.")
else:
    ablation_configs = ALL_ABLATIONS
    ablation_examples = examples

abl_file = ARTIFACTS / "ablation.jsonl"
done_abl = set()
if abl_file.exists():
    with open(abl_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            r = json.loads(line)
            done_abl.add((r["id"], r["config"]))

with open(abl_file, "a", encoding="utf-8") as f:
    for cfg in ablation_configs:
        for ex in ablation_examples:
            key = (ex["id"], cfg["name"])
            if key in done_abl:
                continue
            q = ex["question"] + cfg["prompt_suffix"]
            print(f"[{cfg['name']}] {ex['id']}")
            r = run_vlm(ex["image"], q, max_new_tokens=16,
                        resolution=cfg["resolution"], **cfg["decoding"])
            rec = {"id": ex["id"], "config": cfg["name"], "answer": r["answer"],
                   "latency_s": r["latency_s"], "prompt": q,
                   "resolution": cfg["resolution"]}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            f.flush()

abl_all = [json.loads(l) for l in open(abl_file, encoding="utf-8") if l.strip()]
check(len(abl_all) > 0, f"ablation.jsonl: {len(abl_all)} записей")


[base] pixmo-cap_train_2


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


[base] pixmo-cap_train_3


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


[base] pixmo-cap_train_5


...

[sampling] pixmo-count_validation_15


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


[sampling] pixmo-count_validation_17


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


[sampling] pixmo-count_validation_20
✔ ablation.jsonl: 80 записей


### Качество абляционных вариантов

*Вес: 1 балл.*

**Постановка.** Изменение prompt или разрешения следует сопоставлять по измеримому качеству коротких ответов.

Выполните следующие действия:
1. Для VQA/OCR рассчитайте нормализованный exact match по каждому варианту.
2. Сохраните размер и точность каждой группы.

**Результат.** ablation_quality.json с фактической точностью VQA/OCR.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [15]:
# Реализуйте это задание по постановке выше.
from collections import defaultdict

abl_by_cfg = defaultdict(list)
for r in abl_all:
    abl_by_cfg[r["config"]].append(r)

ex_by_id = {ex["id"]: ex for ex in examples}
em_cats = {"open_vqa", "document_qa", "object_count_json"}

abl_quality = {}
for cfg, rows in abl_by_cfg.items():
    per_cat = defaultdict(list)
    for r in rows:
        ex = ex_by_id.get(r["id"])
        if ex is None or ex["category"] not in em_cats:
            continue
        if ex["category"] == "object_count_json":
            s = count_match(r["answer"], ex["reference"])
        else:
            s = em_score(r["answer"], ex["reference"])
        per_cat[ex["category"]].append(s)

    flat = [s for v in per_cat.values() for s in v]
    abl_quality[cfg] = {
        "by_category": {c: {"n": len(v), "em": float(np.mean(v))} for c, v in per_cat.items()},
        "overall_em": float(np.mean(flat)) if flat else None,
        "n_rows": len(rows),
    }

with open(ARTIFACTS / "ablation_quality.json", "w", encoding="utf-8") as f:
    json.dump(abl_quality, f, ensure_ascii=False, indent=2)
print(json.dumps(abl_quality, ensure_ascii=False, indent=2))
check(bool(abl_quality), "ablation_quality.json записан")

{
  "base": {
    "by_category": {
      "open_vqa": {
        "n": 5,
        "em": 0.0
      },
      "document_qa": {
        "n": 5,
        "em": 0.2
      },
      "object_count_json": {
        "n": 5,
        "em": 0.0
      }
    },
    "overall_em": 0.06666666666666667,
    "n_rows": 20
  },
  "terse": {
    "by_category": {
      "open_vqa": {
        "n": 5,
        "em": 0.0
      },
      "document_qa": {
        "n": 5,
        "em": 0.6
      },
      "object_count_json": {
        "n": 5,
        "em": 0.0
      }
    },
    "overall_em": 0.2,
    "n_rows": 20
  },
  "low_res": {
    "by_category": {
      "open_vqa": {
        "n": 5,
        "em": 0.0
      },
      "document_qa": {
        "n": 5,
        "em": 0.0
      },
      "object_count_json": {
        "n": 5,
        "em": 0.0
      }
    },
    "overall_em": 0.0,
    "n_rows": 20
  },
  "sampling": {
    "by_category": {
      "open_vqa": {
        "n": 5,
        "em": 0.0
      },
      "document_qa": {


### Задержка и сводная оценка

*Вес: 1 балл.*

**Постановка.** Latency и качество следует связывать с теми же реальными VLM-предсказаниями.

Выполните следующие действия:
1. Вычислите median и p95 задержки каждого варианта.
2. Сохраните сводку качества, ручных ошибок и бюджета.

**Результат.** eval_summary.json с фактическими метриками и числом изображений.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [16]:
# Реализуйте это задание по постановке выше.
def lat_stats(rows):
    arr = np.array([r["latency_s"] for r in rows], dtype=float)
    if len(arr) == 0:
        return {"n": 0, "median_s": None, "p95_s": None}
    return {"n": int(len(arr)),
            "median_s": float(np.median(arr)),
            "p95_s":    float(np.quantile(arr, 0.95))}

lat_base   = lat_stats(preds)
lat_by_cfg = {cfg: lat_stats(rows) for cfg, rows in abl_by_cfg.items()}

# Подхватываем актуальный error_report
with open(ARTIFACTS / "error_report.json", encoding="utf-8") as f:
    er = json.load(f)

summary = {
    "model_id": MODEL_ID,
    "quant": QUANT,
    "device": DEVICE,
    "fast_dev_run": FAST_DEV_RUN,
    "n_examples": len(examples),
    "per_task": {t: sum(1 for ex in examples if ex["task"] == t)
                 for t in ("captioning", "vqa", "ocr", "structured")},
    "metrics": metrics,
    "ablation_quality": abl_quality,
    "latency": {"base": lat_base, "by_config": lat_by_cfg},
    "manual_review": {"total": er["total"], "reviewed": er["reviewed"],
                      "complete": er["complete"]},
}
with open(ARTIFACTS / "eval_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(json.dumps(summary, ensure_ascii=False, indent=2))
check(summary["n_examples"] > 0, "eval_summary.json записан")

{
  "model_id": "HuggingFaceTB/SmolVLM2-2.2B-Instruct",
  "quant": "fp32_cpu",
  "device": "cpu",
  "fast_dev_run": false,
  "n_examples": 20,
  "per_task": {
    "captioning": 5,
    "vqa": 5,
    "ocr": 5,
    "structured": 5
  },
  "metrics": {
    "overall": {
      "n": 20,
      "categories": {
        "caption": 5,
        "open_vqa": 5,
        "document_qa": 5,
        "object_count_json": 5
      }
    },
    "by_category": {
      "caption": {
        "n": 5,
        "em": null,
        "note": "EM неприменим, требуется ручная оценка"
      },
      "open_vqa": {
        "n": 5,
        "em": 0.0
      },
      "document_qa": {
        "n": 5,
        "em": 0.2
      },
      "object_count_json": {
        "n": 5,
        "em": 0.0
      }
    }
  },
  "ablation_quality": {
    "base": {
      "by_category": {
        "open_vqa": {
          "n": 5,
          "em": 0.0
        },
        "document_qa": {
          "n": 5,
          "em": 0.2
        },
        "object_count_

### Творческий бонус: проверка формулировки вопроса

*Вес: 1 дополнительный балл.*

**Постановка.** Уточнение вопроса может изменить ответ модели без смены изображения.

Выполните следующие действия:
1. Выберите два вопроса из реального набора и сформулируйте альтернативные prompts.
2. Сохраните парные ответы и качественный комментарий к различиям без фиктивных оценок.

**Результат.** bonus_prompts.json с исходными и новыми ответами.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [17]:
# Реализуйте это задание по постановке выше.
RUN_BONUS = True   # см. README: творческий бонус запускается отдельно

if not RUN_BONUS:
    print("RUN_BONUS=False — пропускаем бонус.")
else:
    # Приоритет — VQA-примеры; если их меньше 2, добираем любыми
    vqa_ex = [ex for ex in examples if ex["task"] == "vqa"]
    if len(vqa_ex) < 2:
        vqa_ex += [ex for ex in examples if ex["task"] != "vqa"]
    vqa_ex = vqa_ex[:2]

    ALT_TEMPLATES = [
        ("brief",     "Answer in one word only: {q}"),
        ("reasoning", "Think step-by-step, then give a one-line answer: {q}"),
    ]

    bonus = []
    for ex in vqa_ex:
        original = preds_by_id.get(ex["id"], {"answer": ""})["answer"]
        for style_name, tmpl in ALT_TEMPLATES:
            alt_q = tmpl.format(q=ex["question"])
            r = run_vlm(ex["image"], alt_q, max_new_tokens=48, do_sample=False)
            changed = r["answer"].strip().lower() != original.strip().lower()
            bonus.append({
                "id": ex["id"],
                "image": ex["image"],
                "original_question": ex["question"],
                "original_answer": original,
                "alternate_style": style_name,
                "alternate_question": alt_q,
                "alternate_answer": r["answer"],
                "changed": changed,
                "qualitative_note": ("Ответ изменился при смене формулировки"
                                     if changed else "Ответ совпал с исходным"),
            })

    with open(ARTIFACTS / "bonus_prompts.json", "w", encoding="utf-8") as f:
        json.dump(bonus, f, ensure_ascii=False, indent=2)
    print(json.dumps(bonus, ensure_ascii=False, indent=2))
    check(len(bonus) >= 2, "bonus_prompts.json записан")


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


[
  {
    "id": "pixmo-ask-model-anything_train_2",
    "image": "D:\\ML_SOTA\\data\\lab06\\pixmo-ask-model-anything-default-train-2.jpg",
    "original_question": "What kind of vehicle is this?",
    "original_answer": "This is a hang glider.",
    "alternate_style": "brief",
    "alternate_question": "Answer in one word only: What kind of vehicle is this?",
    "alternate_answer": "glider",
    "changed": true,
    "qualitative_note": "Ответ изменился при смене формулировки"
  },
  {
    "id": "pixmo-ask-model-anything_train_2",
    "image": "D:\\ML_SOTA\\data\\lab06\\pixmo-ask-model-anything-default-train-2.jpg",
    "original_question": "What kind of vehicle is this?",
    "original_answer": "This is a hang glider.",
    "alternate_style": "reasoning",
    "alternate_question": "Think step-by-step, then give a one-line answer: What kind of vehicle is this?",
    "alternate_answer": "glider plane",
    "changed": true,
    "qualitative_note": "Ответ изменился при смене формулировки"

## Анализ результатов

Сопоставьте измеренные показатели на отложенных реальных данных. Укажите бюджет, ошибки модели и ограничения сокращённого запуска.


## Выводы

Сформулируйте предметный вывод по сохранённым артефактам.
